In [277]:
import jax.numpy as jnp
from scipy.stats import norm, multivariate_normal, binom
from IPython.display import Markdown, display
from jax import random

In [278]:
def to_latex_matrix(matrix):
    """Convert a JAX/numpy matrix to a LaTeX bmatrix string."""
    rows = []
    for row in matrix:
        rows.append(" & ".join(f"{val:.2f}" for val in row))
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"

def to_latex_vector(vector):
    """Convert a JAX/numpy vector to a LaTeX bmatrix column vector."""
    rows = r" \\ ".join(f"{val:.2f}" for val in vector)
    return r"\begin{bmatrix}" + rows + r"\end{bmatrix}"

# Bayesian classification

In [279]:
#Define training data
Xtrain  = jnp.array([[0,0],[1,1]]) 

#Define training targets
ytrain = jnp.array([0,1]) 

# Define test data
Xtest = jnp.array([[0,1]])


######## Parameters ########
alpha = 1/4
D = 2 #Dimension
w_MAP = jnp.array([1,1]) *  0.74077439

######## Functions #########
sigmoid = lambda x: 1 / (1 + jnp.exp(-x))
predict = lambda X,w: sigmoid(w@X.T)


## Likelihood - $p(\bf{y} | \bf{w})$

### Equation

$$p(\bf{y} | \bf{w}) = \prod_{n=1}^{N}{\sigma(\phi(x_{n})^{T} \bf{w})^{y_{n}} (1-\sigma(\phi(x_{n})^{T} w))^{1-y_{n}}}$$

### Code

In [280]:
def likelihood(X, w, use_binom = False):
    if not use_binom: 
        return jnp.prod(predict(X,w)**(ytrain) * (1-predict(X,w))**(1-ytrain))
    return jnp.prod(binom.pmf(ytrain, p = predict(X,w), n = 1))

l = likelihood(Xtrain, w_MAP)
display(Markdown(rf"The likelihood is $p(y | w) = {l:.4f}$"))

The likelihood is $p(y | w) = 0.4074$

## Log-likelihood - $\log{p(\bf{y} | \bf{w})}$

### Equation

$$\log{p(\bf{y} | \bf{w})} = \sum_{n = 1}^{N}{\bf{y_{n}} \log{\sigma(\phi(\bf{x_{n}})^{T} \bf{w})}} + (1-\bf{y_{n}}) \log\left(1- \sigma(\phi(\bf{x_{n}})^{T}\bf{w}) \right)$$

### Code

In [281]:
def log_likelihood(X,w):
    p = predict(X, w)
    return jnp.sum(binom.logpmf(ytrain, p=p, n=1))

ll = log_likelihood(Xtrain, w_MAP)
display(Markdown(rf"The log-likelihood is $\log p(y | w) = {ll:.4f}$"))

The log-likelihood is $\log p(y | w) = -0.8980$

## Joint probability - $p(\bf{y},\bf{w})$

### Equation

$$p(\bf{y}, \bf{w}) = \mathcal{N}(w | 0, \alpha^{-1} \bf{I}) \prod_{n=1}^{N}{\sigma(\phi(x_{n})^{T} \bf{w})^{y_{n}} (1-\sigma(\phi(x_{n})^{T} w))^{1-y_{n}}}$$

### Code

In [282]:
def joint(X, w):
    prior = multivariate_normal.pdf(w, mean=jnp.zeros(D), cov=(1/alpha) * jnp.eye(D))
    return prior * likelihood(X, w)

j = joint(Xtrain, w_MAP)
display(Markdown(rf"The joint probability is $p(y, w) = {j:.4f}$"))

The joint probability is $p(y, w) = 0.0141$

## Log joint probability - $\log{p(\bf{y}, \bf{w})}$

### Equation

$$\log{p(\bf{y} ,\bf{w})} = \log{\mathcal{N}(\bf{w} | \bf{0}, \alpha^{-1} \bf{I})} + \sum_{n = 1}^{N}{\bf{y_{n}} \log{\sigma(\phi(\bf{x_{n}})^{T} \bf{w})}} + (1-\bf{y_{n}}) \log\left(1- \sigma(\phi(\bf{x_{n}})^{T}\bf{w}) \right)$$

In [283]:
def log_joint(X, w):
    log_prior = multivariate_normal.logpdf(w, mean=jnp.zeros(D), cov=(1/alpha) * jnp.eye(D))
    return log_prior+ log_likelihood(X, w)

lj = log_joint(Xtrain, w_MAP)
display(Markdown(rf"The log joint probability is $\log p(y, w) = {lj:.4f}$"))


The log joint probability is $\log p(y, w) = -4.2593$

## Gradient of log joint - $\nabla \log(\bf{y}, \bf{w})$

### Equation 

$$\nabla_w \log p(y, w) = \sum_{n=1}^{N} (y_n - p_n) x_n - \alpha w$$

### Code

In [284]:
def gradient(X,w):
    
    p = predict(X, w)
    err = p - ytrain
    grad = -jnp.sum(err.T*X, axis=0) -alpha*w
    
    return grad

grad = gradient(Xtrain,w=w_MAP[None, :])
grad_latex = to_latex_matrix(grad)
display(Markdown(rf"The gradient is $\nabla_{{w}} \log{{p(y,w_{{MAP}})}} = {grad_latex}$"))

The gradient is $\nabla_{w} \log{p(y,w_{MAP})} = \begin{bmatrix}0.00 & 0.00\end{bmatrix}$

## Hessian of log joint - $\nabla^{2} \log(\bf{y}, \bf{w})$

### Equation

$$\mathcal{H} = -X^\top \text{diag}(v) X - \alpha I$$
where
$$v = p \odot (1 - p), \quad p = \sigma(Xw)$$

### Code

In [285]:
def hessian(X,w):
    p = predict(X, w)
    v = p*(1-p)
    H = -X.T @ jnp.diag(v) @ X -alpha*jnp.identity(D)
    return H

H = hessian(Xtrain, w_MAP)
H_latex = to_latex_matrix(H)
display(Markdown(rf"The hessian is $H = {H_latex}$"))

The hessian is $H = \begin{bmatrix}-0.40 & -0.15 \\ -0.15 & -0.40\end{bmatrix}$

## Laplace approximation of posterior - $p(w | y)$

In [286]:
#Mean of laplace approximation
m = w_MAP
m_latex = to_latex_vector(m)
#Covariance of laplace approximation
S = -jnp.linalg.inv(H)
S_latex = to_latex_matrix(S)

display(Markdown(rf"The laplace approximation is $\mathcal{{N}}\left(w | {m_latex}, {S_latex}\right)$"))

The laplace approximation is $\mathcal{N}\left(w | \begin{bmatrix}0.74 \\ 0.74\end{bmatrix}, \begin{bmatrix}2.91 & -1.09 \\ -1.09 & 2.91\end{bmatrix}\right)$

## Posterior predictive - $p(y^{*} | \bf{w}, \bf{x^{*}})$

In [287]:
mu = Xtest @ m
sigma2 = jnp.diag(Xtest @ S @ Xtest.T)

display(Markdown(rf"The posterior distribution is $p(f | x_*, y) = \mathcal{{N}}(f \mid \mu, \sigma^2)$ where $\mu = {to_latex_vector(mu)}$ and $\sigma^2 = {to_latex_vector(sigma2)}$"))

The posterior distribution is $p(f | x_*, y) = \mathcal{N}(f \mid \mu, \sigma^2)$ where $\mu = \begin{bmatrix}0.74\end{bmatrix}$ and $\sigma^2 = \begin{bmatrix}2.91\end{bmatrix}$

### Probit approximation $\sigma(y) \approx \Phi(y)$

#### Equation

$$\sigma(y) \approx \Phi \left(\frac{\mu}{\sqrt{\frac{8}{\pi} + \sigma^{2}}}\right)$$

#### Code

In [288]:
probit = lambda x: norm.cdf(x)

def probit_approx(mu, sigma):
    return probit(mu / jnp.sqrt(8 / jnp.pi + sigma))

p_approx = probit_approx(mu = mu, sigma=sigma2)
display(Markdown(rf"The predictive probability is $p(y=1 \mid \bf{{x_*}}, y) = \Phi\left(\frac{{\mu}}{{\sqrt{{8/\pi + \sigma^2}}}}\right) = {to_latex_vector(p_approx)}$"))

The predictive probability is $p(y=1 \mid \bf{x_*}, y) = \Phi\left(\frac{\mu}{\sqrt{8/\pi + \sigma^2}}\right) = \begin{bmatrix}0.62\end{bmatrix}$

### Monte Carlo approximation

#### Equation

$$p(y^* = 1 \mid y, x^*) \approx \frac{1}{S} \sum_{i=1}^{S} \sigma\left(f^{(i)}\right)$$

#### Code

In [289]:
def mc_approx(num_samples,mu,sigma):
    key = random.PRNGKey(0)
    f = mu  + jnp.sqrt(sigma)*random.normal(key, shape=(num_samples, len(Xtest)))
    p = sigmoid(f).mean(0)
    return p

p_mc = mc_approx(num_samples=10000, mu = mu, sigma = sigma2)
display(Markdown(rf"The predictive probability is $p(y=1 \mid \bf{{x_*}}, y) = \frac{{1}}{{S}} \sum_{{i=1}}^{{S}} \sigma\left(f^{{(i)}}\right) = {to_latex_vector(p_mc)}$"))

The predictive probability is $p(y=1 \mid \bf{x_*}, y) = \frac{1}{S} \sum_{i=1}^{S} \sigma\left(f^{(i)}\right) = \begin{bmatrix}0.62\end{bmatrix}$